# Hyperband with Optuna

Notebook for the course [Master Hyperparameter Optimization for Tabular Learning](http://www.trainindata.com/p/master-hyperparameter-optimization-for-tabular-learning)

In this notebook, we'll implement [hyperband](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.pruners.HyperbandPruner.html) with Optuna.

In [1]:
import optuna

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import xgboost as xgb

# XGBoostPruningCallback moved out of optuna itself and into the
# separate optuna-integration package (pip install optuna-integration)
from optuna_integration import XGBoostPruningCallback

In [2]:
# load dataset and prepare data

data, target = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.25)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

N_BOOST_ROUNDS = 81

## Define the objective function

Check out [xgboost pruning integration](https://optuna-integration.readthedocs.io/en/stable/reference/generated/optuna_integration.XGBoostPruningCallback.html#optuna_integration.XGBoostPruningCallback)

Taken from https://github.com/optuna/optuna-examples/blob/main/xgboost/xgboost_integration.py

<div style="padding: 12px 16px; border-left: 5px solid #2196f3; background-color: #eaf4fd; border-radius: 4px;">
<strong>Notes:</strong> <code>max_resource</code> must match the available boosting rounds. Pruning and optimization both use ROC AUC. For real model selection, prune on a validation set and reserve the test set for one final evaluation.
</div>

In [3]:
def objective(trial):

    # hyperparameter space
    param = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
    }

    # conditional space: some hyperparams depend on other hyperparams
    if param["booster"] == "gbtree" or param["booster"] == "dart":
        param["max_depth"] = trial.suggest_int("max_depth", 1, 9)
        param["eta"] = trial.suggest_float("eta", 1e-8, 1.0, log=True)
        param["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)
        param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])
    
    if param["booster"] == "dart":
        param["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
        param["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
        param["rate_drop"] = trial.suggest_float("rate_drop", 1e-8, 1.0, log=True)
        param["skip_drop"] = trial.suggest_float("skip_drop", 1e-8, 1.0, log=True)

    # Add a callback for pruning.
    # Hyperband evaluates ROC AUC after each boosting round.
    pruning_callback = XGBoostPruningCallback(trial, "validation-auc")
    
    # set up the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=N_BOOST_ROUNDS,
        evals=[(dtest, "validation")],
        callbacks=[pruning_callback],
    )
    
    # evaluate
    preds = bst.predict(dtest)
    roc_auc = roc_auc_score(y_test, preds)
    
    return roc_auc

In [4]:
study = optuna.create_study(
    
    # a way to sample hyperparameters to create the configurations
    sampler=optuna.samplers.RandomSampler(),
    
    pruner=optuna.pruners.HyperbandPruner(
        
        # controls the minimum validation rounds that it needs to wait before stopping
        min_resource=1,
        
        # the maximum budget that we have,
        max_resource=N_BOOST_ROUNDS,
        
        # how many configurations to promote to the next round
        reduction_factor=3,
    ),
    
    direction="maximize",

)

study.optimize(
    objective, 
    
    # this parameter limits the number of configurations that
    # will be examined
    n_trials=50,
)

[I 2026-08-26 13:58:17,058] A new study created in memory with name: no-name-282934ab-b9da-441b-8146-2b0bf1a59395


[0]	validation-auc:0.97059
[1]	validation-auc:0.97208
[2]	validation-auc:0.97485
[3]	validation-auc:0.97379
[4]	validation-auc:0.97613
[5]	validation-auc:0.97783
[6]	validation-auc:0.97826
[7]	validation-auc:0.97933
[8]	validation-auc:0.97954
[9]	validation-auc:0.98018
[10]	validation-auc:0.98061
[11]	validation-auc:0.98188
[12]	validation-auc:0.98231
[13]	validation-auc:0.98274
[14]	validation-auc:0.98274
[15]	validation-auc:0.98338
[16]	validation-auc:0.98380
[17]	validation-auc:0.98423
[18]	validation-auc:0.98572
[19]	validation-auc:0.98572
[20]	validation-auc:0.98615
[21]	validation-auc:0.98636
[22]	validation-auc:0.98700
[23]	validation-auc:0.98764
[24]	validation-auc:0.98743
[25]	validation-auc:0.98828
[26]	validation-auc:0.98849
[27]	validation-auc:0.98913
[28]	validation-auc:0.99041
[29]	validation-auc:0.99126
[30]	validation-auc:0.99169
[31]	validation-auc:0.99211
[32]	validation-auc:0.99254
[33]	validation-auc:0.99275
[34]	validation-auc:0.99318
[35]	validation-auc:0.99382
[3

[I 2026-08-26 13:58:17,098] Trial 0 finished with value: 0.9978687127024722 and parameters: {'booster': 'gblinear', 'lambda': 5.184217496286076e-08, 'alpha': 0.0022085603750340646}. Best is trial 0 with value: 0.9978687127024722.


[0]	validation-auc:0.94395
[1]	validation-auc:0.95567
[2]	validation-auc:0.96014


[I 2026-08-26 13:58:17,100] Trial 1 pruned. Trial was pruned at iteration 3.


...

[I 2026-08-26 13:58:18,396] Trial 47 pruned. Trial was pruned at iteration 3.


[0]	validation-auc:0.98870


[I 2026-08-26 13:58:18,399] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	validation-auc:0.98806
[1]	validation-auc:0.99787
[2]	validation-auc:0.99787


[I 2026-08-26 13:58:18,409] Trial 49 pruned. Trial was pruned at iteration 3.


In [5]:
# the best hyperparameters

study.best_params

{'booster': 'gbtree',
 'lambda': 0.000198564408123657,
 'alpha': 0.0002698407146158256,
 'max_depth': 5,
 'eta': 0.1752414992922432,
 'gamma': 0.04672511647709466,
 'grow_policy': 'depthwise'}

In [6]:
# the best performance value

study.best_value

0.9991474850809889

In [7]:
r = study.trials_dataframe()

r

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_booster,params_eta,params_gamma,params_grow_policy,...,params_max_depth,params_normalize_type,params_rate_drop,params_sample_type,params_skip_drop,system_attrs_completed_rung_0,system_attrs_completed_rung_1,system_attrs_completed_rung_2,system_attrs_completed_rung_3,state
0,0,0.997869,2026-08-26 13:58:17.058364,2026-08-26 13:58:17.098194,0 days 00:00:00.039830,2.208560e-03,gblinear,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.973785,0.980179,0.989130,NaN,COMPLETE
1,1,0.960784,2026-08-26 13:58:17.098499,2026-08-26 13:58:17.100844,0 days 00:00:00.002345,2.481520e-07,gblinear,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.960784,NaN,NaN,NaN,PRUNED
2,2,0.997869,2026-08-26 13:58:17.101035,2026-08-26 13:58:17.245206,0 days 00:00:00.144171,7.659215e-05,gbtree,3.942942e-04,6.431941e-03,depthwise,...,8.0,NaN,NaN,NaN,NaN,0.997442,0.997656,0.998082,0.997869,COMPLETE
3,3,0.891091,2026-08-26 13:58:17.245441,2026-08-26 13:58:17.248328,0 days 00:00:00.002887,1.214084e-07,dart,1.545767e-07,1.056264e-08,lossguide,...,1.0,forest,7.351108e-01,uniform,2.036885e-05,0.891091,NaN,NaN,NaN,PRUNED
4,4,0.959719,2026-08-26 13:58:17.248502,2026-08-26 13:58:17.249535,0 days 00:00:00.001033,4.710531e-01,gblinear,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.959719,NaN,NaN,NaN,PRUNED
5,5,0.995631,2026-08-26 13:58:17.249714,2026-08-26 13:58:17.254216,0 days 00:00:00.004502,3.133644e-01,gbtree,4.443815e-01,4.981134e-06,lossguide,...,5.0,NaN,NaN,NaN,NaN,0.995631,NaN,NaN,NaN,PRUNED
6,6,0.998721,2026-08-26 13:58:17.254394,2026-08-26 13:58:17.339768,0 days 00:00:00.085374,2.448839e-07,dart,5.146885e-01,2.499966e-06,lossguide,...,8.0,tree,9.506749e-03,weighted,1.056949e-03,1.000000,0.998295,1.000000,0.998934,COMPLETE
7,7,0.999147,2026-08-26 13:58:17.339974,2026-08-26 13:58:17.417087,0 days 00:00:00.077113,2.698407e-04,gbtree,1.752415e-01,4.672512e-02,depthwise,...,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
8,8,0.997869,2026-08-26 13:58:17.417293,2026-08-26 13:58:17.423614,0 days 00:00:00.006321,3.157831e-07,dart,2.023724e-07,3.518419e-06,lossguide,...,5.0,forest,4.693502e-05,weighted,2.164255e-08,0.997869,NaN,NaN,NaN,PRUNED
9,9,0.965899,2026-08-26 13:58:17.423783,2026-08-26 13:58:17.424961,0 days 00:00:00.001178,1.081496e-06,gblinear,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.965899,NaN,NaN,NaN,PRUNED


In [12]:
r["boosting_rounds"] = [
    trial.last_step + 1 if trial.last_step is not None else None
    for trial in study.trials
]

r.loc[r["state"] == "COMPLETE", "boosting_rounds"]

0     81
2     81
6     81
7     81
10    81
11    81
14    81
20    81
38    81
Name: boosting_rounds, dtype: int64

In [8]:
# the number of brackets evaluated

study.pruner._n_brackets

5

In [9]:
# budget allocated to each initial configuration
# within each bracket

study.pruner._trial_allocation_budgets

[81, 34, 15, 8, 5]

## Understanding the Hyperband brackets

The values `[81, 34, 15, 8, 5]` are **trial-allocation weights**. They determine the approximate proportion of trials assigned to each bracket.

With `min_resource=1`, `max_resource=81`, and `reduction_factor=3`, Hyperband creates the following brackets:

| Bracket | Allocation weight | First pruning checkpoint | Subsequent checkpoints |
|:---:|---:|---:|:---|
| 0 | 81 | 1 | 3, 9, 27, 81 |
| 1 | 34 | 3 | 9, 27, 81 |
| 2 | 15 | 9 | 27, 81 |
| 3 | 8 | 27 | 81 |
| 4 | 5 | 81 | None |

Bracket 0 receives the most trials and may prune them very early. Later brackets receive fewer trials but protect them from pruning for longer, allowing slowly improving configurations a chance to succeed. At each checkpoint, successive halving generally retains approximately the best `1 / reduction_factor`&mdash;one-third here&mdash;of the comparable trials in that bracket.

For 50 trials, the expected allocation is approximately 28, 12, 5, 3, and 2 trials across brackets 0 to 4. The exact counts may differ because Optuna uses hash-based bracket assignment.

> **Important:** Every XGBoost trial physically starts at boosting round 1. A trial assigned to bracket 2 still trains rounds 1 through 9; Hyperband simply does not consider pruning it before its first checkpoint. Bracket 4 effectively trains for the full resource without early pruning.

XGBoost reports iterations using zero-based numbering, so 81 boosting rounds correspond to reported steps 0 to 80. Consequently, `trial.last_step + 1` gives the number of completed boosting rounds.


## Implementation notes

- Optuna assigns trials to brackets using a deterministic hash of the study name and trial number, weighted by each bracket's allocation budget.
- Bracket IDs are calculated internally and are not included as a public column in `study.trials_dataframe()`.
- Attributes beginning with an underscore, such as `_n_brackets`, are private implementation details and may change.
